# Exploratory analysis
Dataset overlap, rating distributions and the α(u) schedule. Run `python -m recsys.cli extract align` first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from recsys.config import Config
from recsys.data import loaders
from recsys.data.align import coverage_report
from recsys.models.hybrid import DynamicHybridRecommender

cfg = Config()
items = pd.read_parquet(cfg.paths.items)
ratings = loaders.load_ml_ratings(cfg.paths)
coverage_report(items)

## Catalogue overlap
Share of MovieLens movies with a CMU plot, by match pass and by rating volume (popular movies are better covered).

In [ ]:
pop = ratings.groupby('movieId').size().rename('n_ratings')
df = items.join(pop, on='movieId')
df['pop_bucket'] = pd.qcut(df['n_ratings'], 5, labels=['q1 (rare)', 'q2', 'q3', 'q4', 'q5 (popular)'])
print('share of *ratings* on movies with a plot:', round(df.loc[df.has_plot, 'n_ratings'].sum() / df['n_ratings'].sum(), 4))
df.groupby('pop_bucket', observed=True)['has_plot'].mean().plot.bar(title='Plot coverage by popularity quintile', ylim=(0, 1));

## Ratings

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ratings['rating'].value_counts().sort_index().plot.bar(ax=ax[0], title='Rating distribution')
ratings.groupby('userId').size().plot.hist(bins=60, ax=ax[1], log=True, title='Ratings per user (N_u)')
ratings.groupby('userId').size().describe()

## α(u) schedule
Every ML-1M user has ≥ 20 ratings, so on the full data α(u) sits near α_max; the cold-start simulation (`python -m recsys.cli coldstart`) truncates histories to exercise the ramp.

In [ ]:
n = np.arange(0, 41)
for g in (0.05, 0.15, 0.3):
    plt.plot(n, DynamicHybridRecommender(0.8, g).calculate_alpha(n), label=f'γ={g}')
plt.axhline(0.8, ls=':', c='grey'); plt.xlabel('N_u'); plt.ylabel('α(u)'); plt.legend(); plt.title('α(u) = α_max (1 − e^(−γ N_u))');

## Evaluation results
Produced by `python -m recsys.cli evaluate` / `coldstart`.

In [ ]:
res = pd.read_csv(cfg.paths.artifacts / 'eval_results.csv')
res.pivot(index='model', columns='split', values='ndcg@10').sort_values('random')

In [ ]:
cs = pd.read_csv(cfg.paths.artifacts / 'coldstart_results.csv')
cs.pivot(index='n_u', columns='model', values='ndcg@10').loc[['0', '3', '5', '10', '20', 'all']].plot(marker='o', title='NDCG@10 vs history length');